# Decoder Training — Flux-Apex-V1.flx

**Large-Scale Decoder Training with Google Drive Checkpoints**

## Key Features
- **Loads** Flux-Apex-V1.flx from HuggingFace
- **Trains** on 50,000 curated examples (6 diverse datasets)
- **Saves** .pt checkpoints to Google Drive every 5k steps
- **Uploads** final .flx to HuggingFace only at end
- **Word-boundary aware** + repetition penalty

## Datasets (~8k each)
| Category | Dataset | Purpose |
|----------|---------|--------|
| WebText | OpenWebText | General web text (replay buffer) |
| Assistancy | OpenAssistant | Helpful responses |
| Dialogues | Anthropic HH-RLHF | Natural conversations |
| Q&A | SQuAD | Question answering |
| Storytelling | WritingPrompts | Narrative coherence |
| Reasoning | GSM8K | Step-by-step logic |

## Hardware
- **Target:** Colab T4/A100 or Kaggle T4
- **Runtime:** ~3-4 hours
- **Checkpoint:** Every 5k steps → Google Drive
- **Final:** Upload to HuggingFace

---

## Cell 1: Setup Storage (Auto-detect Colab/Kaggle)

In [ ]:
# Auto-detect environment and setup checkpoint storage
import os
from pathlib import Path

# Detect environment
IS_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
IS_KAGGLE = os.path.exists('/kaggle')

if IS_COLAB:
    # Try to mount Google Drive (Colab)
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        GDRIVE_BASE = Path('/content/drive/MyDrive')
        CHECKPOINT_STORAGE = GDRIVE_BASE / 'FLUX_checkpoints'
        CHECKPOINT_STORAGE.mkdir(parents=True, exist_ok=True)
        print(f'  ✓ Google Drive mounted')
        print(f'  ✓ Checkpoints save to: {CHECKPOINT_STORAGE}')
    except Exception as e:
        print(f'  ⚠ Google Drive mount failed: {e}')
        print(f'  → Using local storage instead')
        CHECKPOINT_STORAGE = Path('/content/checkpoints')
        CHECKPOINT_STORAGE.mkdir(parents=True, exist_ok=True)

elif IS_KAGGLE:
    # Kaggle: save to working directory (auto-saved as output)
    CHECKPOINT_STORAGE = Path('/kaggle/working/checkpoints')
    CHECKPOINT_STORAGE.mkdir(parents=True, exist_ok=True)
    print(f'  ✓ Kaggle detected')
    print(f'  ✓ Checkpoints save to: {CHECKPOINT_STORAGE}')
    print(f'  → Will be available in Output after session')

else:
    # Local development
    CHECKPOINT_STORAGE = Path('./checkpoints')
    CHECKPOINT_STORAGE.mkdir(parents=True, exist_ok=True)
    print(f'  ✓ Local environment')
    print(f'  ✓ Checkpoints save to: {CHECKPOINT_STORAGE}')

print(f'\n  Environment: {"Colab" if IS_COLAB else "Kaggle" if IS_KAGGLE else "Local"}')

## Cell 2: Clone/Pull FLUX Repository

In [ ]:
%%time
import os
from pathlib import Path

REPO_URL = 'https://github.com/Unseengap/FLUX.git'
ROOT = Path('/kaggle/working/FLUX') if Path('/kaggle').exists() else Path('/content/FLUX')

if ROOT.exists():
    print('  Pulling latest...')
    !cd {ROOT} && git pull
else:
    print('  Cloning repo...')
    !git clone {REPO_URL} {ROOT}

os.chdir(ROOT)
print(f'  Working dir: {os.getcwd()}')

# Install dependencies
!pip install -q -e . 2>/dev/null || pip install -q -r requirements.txt
!pip install -q huggingface_hub datasets transformers

print('  ✓ Dependencies installed')

## Cell 3: Setup Paths & Logger

In [ ]:
import sys
from pathlib import Path

ROOT = Path('/kaggle/working/FLUX') if Path('/kaggle').exists() else Path('/content/FLUX')
PHASES_DIR = ROOT / 'phases'
CHECKPOINT_DIR = ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Add all phase paths
for phase_name in ['phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase6', 'phase7', 'phase8', 'phase8_8', 'phase8_9', 'phase9']:
    p = PHASES_DIR / phase_name
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from flux_utils import PhaseLogger, get_device, HF_REPO_ID, _resolve_hf_token

log = PhaseLogger(phase=14)  # Decoder training phase
log.separator("Decoder Training 50k")

print('  ✓ Paths configured')
print(f'  ROOT: {ROOT}')
print(f'  CHECKPOINT_DIR: {CHECKPOINT_DIR}')
print(f'  CHECKPOINT_STORAGE: {CHECKPOINT_STORAGE}')

## Cell 4: Hardware & Secrets

In [ ]:
import torch
import gc

log.cell_start("Cell 4 — Hardware & Secrets")

device = get_device()
print(f'  Device: {device}')

if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU: {gpu_name} ({vram:.1f} GB)')

# Load HF token (for final upload only)
hf_token = _resolve_hf_token()
if hf_token:
    print('  ✓ HF_TOKEN loaded (for final upload)')
else:
    print('  ⚠ HF_TOKEN not found — final upload will fail')

log.cell_end("Cell 4 — Hardware & Secrets", "PASS")

## Cell 5: Download & Load Flux-Apex-V1.flx

In [ ]:
from huggingface_hub import hf_hub_download
from datetime import datetime

log.cell_start("Cell 5 — Load Flux-Apex-V1.flx")

FLX_NAME = 'Flux-Apex-V1.flx'
FLX_HF_PATH = f'checkpoints/{FLX_NAME}'
flx_path = CHECKPOINT_DIR / FLX_NAME

# Download from HuggingFace
print(f'  Downloading {FLX_NAME} from HuggingFace...')
try:
    downloaded = hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=FLX_HF_PATH,
        local_dir=str(ROOT),
        token=hf_token,
        force_download=False,
    )
    size_mb = Path(downloaded).stat().st_size / 1e6
    print(f'  ✓ Downloaded {FLX_NAME} ({size_mb:.1f} MB)')
except Exception as e:
    if flx_path.exists():
        print(f'  ⚠ Download failed, using local: {e}')
    else:
        raise RuntimeError(f"Cannot get {FLX_NAME}: {e}")

# Load the full archive
print(f'\n  Loading {FLX_NAME}...')
apex = torch.load(str(flx_path), map_location='cpu', weights_only=False)

print(f'  ✓ Loaded: {FLX_NAME}')
print(f'    Version: {apex.get("version", "unknown")}')
print(f'    Phase: {apex.get("phase", "unknown")}')

# Check decoder exists
if 'decoder' not in apex:
    raise KeyError("No decoder found in Flux-Apex-V1.flx!")

decoder_info = apex['decoder']
decoder_config = decoder_info.get('config', {})
print(f'\n  Decoder config:')
print(f'    Hidden dim: {decoder_config.get("hidden_dim", 1024)}')
print(f'    Layers: {decoder_config.get("num_layers", 4)}')

log.cell_end("Cell 5 — Load Flux-Apex-V1.flx", "PASS")

## Cell 6: Build WaveDecoder from Apex State

In [ ]:
log.cell_start("Cell 6 — Build WaveDecoder")

# Import WaveDecoder
from wave_decoder import WaveDecoder
from cse import ContinuousSemanticEncoder

# Build CSE (frozen, for encoding)
cse = ContinuousSemanticEncoder()
cse.load_state_dict(apex['cse']['state_dict'])
cse = cse.to(device).eval()
for p in cse.parameters():
    p.requires_grad = False
print(f'  ✓ CSE loaded and frozen')

# Build WaveDecoder
decoder_config = apex['decoder'].get('config', {
    'wave_dim': 432,
    'field_features': 512,
    'embed_dim': 256,
    'hidden_dim': 1024,
    'num_layers': 4,
    'num_heads': 16,
    'vocab_size': 256,
})

decoder = WaveDecoder(
    wave_dim=decoder_config.get('wave_dim', 432),
    field_features=decoder_config.get('field_features', 512),
    embed_dim=decoder_config.get('embed_dim', 256),
    hidden_dim=decoder_config.get('hidden_dim', 1024),
    num_layers=decoder_config.get('num_layers', 4),
    num_heads=decoder_config.get('num_heads', 16),
    vocab_size=decoder_config.get('vocab_size', 256),
)

# Load weights
decoder_state = apex['decoder']['state_dict']
cleaned_state = {k.replace('_orig_mod.', ''): v for k, v in decoder_state.items()}
decoder.load_state_dict(cleaned_state)
decoder = decoder.to(device)

n_params = sum(p.numel() for p in decoder.parameters())
print(f'  ✓ WaveDecoder loaded: {n_params:,} parameters')

# Get field features for context
field_state = apex['field']['state_dict']['state']
field_mean = field_state.mean(dim=(0, 1, 2))
print(f'  ✓ Field mean computed: {field_mean.shape}')

log.cell_end("Cell 6 — Build WaveDecoder", "PASS")

## Cell 7: Load Diverse Datasets (50k total)

In [ ]:
log.cell_start("Cell 7 — Load Diverse Datasets")

from datasets import load_dataset
import random

random.seed(42)

print("\n  Loading 6 diverse datasets (~8k each, 50k total)...")
print("  " + "="*50)

all_texts = []
dataset_sources = {}

def clear_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ─────────────────────────────────────────────
# 1. OPENWEBTEXT (10k) — General web text (replay buffer)
# ─────────────────────────────────────────────
print("\n  [1/6] OpenWebText (replay buffer)...")
try:
    owt = load_dataset("openwebtext", split="train", streaming=True)
    web_texts = []
    for i, item in enumerate(owt):
        text = item.get('text', '')
        if text and 100 < len(text) < 600:
            web_texts.append(text[:500])
        if len(web_texts) >= 10000:
            break
    all_texts.extend(web_texts)
    dataset_sources['openwebtext'] = len(web_texts)
    del owt
    clear_mem()
    print(f"       ✓ {len(web_texts)} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    dataset_sources['openwebtext'] = 0

# ─────────────────────────────────────────────
# 2. ASSISTANCY (8k) — Helpful responses
# ─────────────────────────────────────────────
print("\n  [2/6] Assistancy (OpenAssistant)...")
try:
    oasst = load_dataset("OpenAssistant/oasst1", split="train")
    assist_texts = []
    for item in oasst:
        if item.get('text') and 50 < len(item['text']) < 500:
            assist_texts.append(item['text'])
    assist_texts = random.sample(assist_texts, min(8000, len(assist_texts)))
    all_texts.extend(assist_texts)
    dataset_sources['assistancy'] = len(assist_texts)
    del oasst
    clear_mem()
    print(f"       ✓ {len(assist_texts)} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    dataset_sources['assistancy'] = 0

# ─────────────────────────────────────────────
# 3. DIALOGUES (8k) — Natural conversations
# ─────────────────────────────────────────────
print("\n  [3/6] Dialogues (Anthropic HH-RLHF)...")
try:
    hh = load_dataset("Anthropic/hh-rlhf", split="train")
    dialog_texts = []
    for item in hh:
        text = item.get('chosen', '')
        if text and 100 < len(text) < 800:
            dialog_texts.append(text)
        if len(dialog_texts) >= 10000:
            break
    dialog_texts = random.sample(dialog_texts, min(8000, len(dialog_texts)))
    all_texts.extend(dialog_texts)
    dataset_sources['dialogues'] = len(dialog_texts)
    del hh
    clear_mem()
    print(f"       ✓ {len(dialog_texts)} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    dataset_sources['dialogues'] = 0

# ─────────────────────────────────────────────
# 4. Q&A (8k) — Question answering
# ─────────────────────────────────────────────
print("\n  [4/6] Q&A (SQuAD)...")
try:
    squad = load_dataset("squad", split="train")
    qa_texts = []
    for item in squad:
        q = item.get('question', '')
        a = item.get('answers', {}).get('text', [''])[0] if item.get('answers') else ''
        if q and a:
            text = f"Question: {q}\nAnswer: {a}"
            if 30 < len(text) < 400:
                qa_texts.append(text)
    qa_texts = random.sample(qa_texts, min(8000, len(qa_texts)))
    all_texts.extend(qa_texts)
    dataset_sources['qa'] = len(qa_texts)
    del squad
    clear_mem()
    print(f"       ✓ {len(qa_texts)} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    dataset_sources['qa'] = 0

# ─────────────────────────────────────────────
# 5. STORYTELLING (8k) — Narrative coherence
# ─────────────────────────────────────────────
print("\n  [5/6] Storytelling (WritingPrompts)...")
try:
    wp = load_dataset("euclaise/writingprompts", split="train")
    story_texts = []
    for item in wp:
        story = item.get('story', '') or item.get('text', '')
        if story and 100 < len(story) < 800:
            story_texts.append(story[:600])
    story_texts = random.sample(story_texts, min(8000, len(story_texts)))
    all_texts.extend(story_texts)
    dataset_sources['storytelling'] = len(story_texts)
    del wp
    clear_mem()
    print(f"       ✓ {len(story_texts)} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    # Fallback to Dolly
    try:
        dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
        story_texts = []
        for item in dolly:
            text = f"{item.get('instruction', '')}\n{item.get('response', '')}"
            if 100 < len(text) < 800:
                story_texts.append(text)
        story_texts = random.sample(story_texts, min(8000, len(story_texts)))
        all_texts.extend(story_texts)
        dataset_sources['storytelling'] = len(story_texts)
        del dolly
        clear_mem()
        print(f"       ✓ {len(story_texts)} samples (Dolly fallback)")
    except Exception as e2:
        print(f"       ⚠ Fallback failed: {e2}")
        dataset_sources['storytelling'] = 0

# ─────────────────────────────────────────────
# 6. REASONING (8k) — Step-by-step logic
# ─────────────────────────────────────────────
print("\n  [6/6] Reasoning (GSM8K)...")
try:
    gsm = load_dataset("gsm8k", "main", split="train")
    reason_texts = []
    for item in gsm:
        q = item.get('question', '')
        a = item.get('answer', '')
        if q and a:
            text = f"Problem: {q}\n\nSolution: {a}"
            if len(text) < 800:
                reason_texts.append(text)
    # GSM8K has ~7.5k samples, take all
    reason_texts = random.sample(reason_texts, min(8000, len(reason_texts)))
    all_texts.extend(reason_texts)
    dataset_sources['reasoning'] = len(reason_texts)
    del gsm
    clear_mem()
    print(f"       ✓ {len(reason_texts)} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    dataset_sources['reasoning'] = 0

# Shuffle all texts
random.shuffle(all_texts)

print("\n  " + "="*50)
print("  DATASET SUMMARY:")
total = 0
for name, count in dataset_sources.items():
    print(f"    {name}: {count}")
    total += count
print(f"  {'─'*30}")
print(f"    TOTAL: {total} samples")
print("  " + "="*50)

log.metric("Total samples", total)
log.cell_end("Cell 7 — Load Diverse Datasets", "PASS")

## Cell 8: Training Configuration

In [ ]:
log.cell_start("Cell 8 — Training Configuration")

import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Training config
BATCH_SIZE = 1
GRAD_ACCUM = 4  # Effective batch = 4
MAX_SEQ_LEN = 384
LR = 2e-4
CHECKPOINT_EVERY = 5000  # Save to GDrive every 5k steps
LOG_EVERY = 500  # Print progress every 500 steps

# Optimizer
optimizer = AdamW(decoder.parameters(), lr=LR, weight_decay=0.01)

# Scheduler
total_steps = len(all_texts) // GRAD_ACCUM
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=LR/10)

print(f"  Training Configuration:")
print(f"    Total samples: {len(all_texts)}")
print(f"    Total steps: {total_steps}")
print(f"    Gradient accumulation: {GRAD_ACCUM}")
print(f"    Max seq length: {MAX_SEQ_LEN} bytes")
print(f"    Learning rate: {LR}")
print(f"    Log every: {LOG_EVERY} steps")
print(f"    Checkpoint every: {CHECKPOINT_EVERY} steps")
print(f"    Estimated checkpoints: {total_steps // CHECKPOINT_EVERY}")

log.cell_end("Cell 8 — Training Configuration", "PASS")

## Cell 9: Checkpoint Functions (Google Drive)

In [ ]:
log.cell_start("Cell 9 — Checkpoint Functions")

from huggingface_hub import HfApi

def save_checkpoint_to_gdrive(decoder, optimizer, scheduler, step, metrics):
    """
    Save training checkpoint (.pt) to Google Drive.
    """
    checkpoint = {
        'step': step,
        'decoder_state_dict': decoder.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'metrics': metrics,
        'timestamp': datetime.now().isoformat(),
    }
    
    path = CHECKPOINT_STORAGE / f'decoder_step_{step}.pt'
    torch.save(checkpoint, str(path))
    size_mb = path.stat().st_size / 1e6
    print(f"    ✓ Checkpoint saved: {path.name} ({size_mb:.1f} MB)")
    return path


def save_final_to_apex_and_upload(decoder, apex_dict, metrics, local_path, hf_token):
    """
    Save final decoder to Apex archive and upload to HuggingFace.
    """
    # Update decoder in apex
    apex_dict['decoder']['state_dict'] = decoder.state_dict()
    apex_dict['decoder']['config']['training_steps'] = metrics.get('total_steps', 0)
    
    # Update metadata
    apex_dict['metadata'] = apex_dict.get('metadata', {})
    apex_dict['metadata']['decoder_training_50k'] = datetime.now().isoformat()
    apex_dict['metadata']['decoder_training_metrics'] = metrics
    apex_dict['modified'] = True
    
    # Save locally
    torch.save(apex_dict, str(local_path))
    size_mb = local_path.stat().st_size / 1e6
    print(f"    ✓ Saved locally: {local_path.name} ({size_mb:.1f} MB)")
    
    # Also save to storage
    storage_flx = CHECKPOINT_STORAGE / local_path.name
    torch.save(apex_dict, str(storage_flx))
    print(f"    ✓ Saved to storage: {storage_flx}")
    
    # Upload to HuggingFace
    if hf_token:
        try:
            api = HfApi(token=hf_token)
            api.upload_file(
                path_or_fileobj=str(local_path),
                path_in_repo=f'checkpoints/{local_path.name}',
                repo_id=HF_REPO_ID,
                commit_message=f'Decoder 50k training — final loss={metrics.get("avg_loss", 0):.4f}',
            )
            print(f"    ✓ Uploaded to HuggingFace Hub")
        except Exception as e:
            print(f"    ⚠ HF upload failed: {e}")
    
    return True


def test_generation(decoder, cse, field_mean, prompt, max_len=60):
    """Quick generation test with repetition penalty."""
    decoder.eval()
    with torch.no_grad():
        wave = cse.encode(prompt)
        wave_seq = wave.full.to(device)
        output = decoder.generate(
            wave_sequence=wave_seq,
            field_features=field_mean.to(device),
            max_length=max_len,
            temperature=0.8,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
        )
    decoder.train()
    return output.decode('utf-8', errors='replace')

print("  ✓ Checkpoint functions defined")
print(f"  ✓ Checkpoints save to: {CHECKPOINT_STORAGE}")
log.cell_end("Cell 9 — Checkpoint Functions", "PASS")

## Cell 10: Training Loop (50k samples)

In [ ]:
log.cell_start("Cell 10 — Training Loop")

import time

print("\n" + "="*60)
print("  DECODER TRAINING — 50k SAMPLES")
print("  Checkpoints: Google Drive every 5k steps")
print("  Final upload: HuggingFace at end")
print("="*60 + "\n")

# Training state
global_step = 0
accum_loss = 0.0
total_loss = 0.0
best_loss = float('inf')
start_time = time.time()

# Word boundary weighting
WORD_BOUNDARY_WEIGHT = 2.0

decoder.train()
optimizer.zero_grad()

for idx, text in enumerate(all_texts):
    try:
        # Encode text to bytes
        text_bytes = text.encode('utf-8', errors='replace')[:MAX_SEQ_LEN]
        if len(text_bytes) < 10:
            continue
        
        target = torch.tensor(list(text_bytes), dtype=torch.long, device=device)
        
        # Encode with CSE (frozen)
        with torch.no_grad():
            wave = cse.encode(text[:MAX_SEQ_LEN])
            wave_seq = wave.full.to(device)
        
        # Forward pass
        logits = decoder(
            target_bytes=target,
            wave_sequence=wave_seq,
            field_features=field_mean.to(device),
        )
        
        # Word-boundary weighted loss
        weights = torch.ones(len(target), device=device)
        weights[target == 32] = WORD_BOUNDARY_WEIGHT  # Space = word boundary
        
        loss_per_token = F.cross_entropy(logits, target, reduction='none')
        loss = (loss_per_token * weights).mean()
        
        # Backward
        loss = loss / GRAD_ACCUM
        loss.backward()
        accum_loss += loss.item() * GRAD_ACCUM
        
        # Step optimizer
        if (idx + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(decoder.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
            global_step += 1
            total_loss += accum_loss
            avg_loss = total_loss / global_step
            
            # Progress update every LOG_EVERY steps
            if global_step % LOG_EVERY == 0:
                elapsed = time.time() - start_time
                rate = global_step / elapsed
                eta = (total_steps - global_step) / rate / 60
                print(f"  Step {global_step}/{total_steps}: loss={accum_loss:.4f} avg={avg_loss:.4f} lr={scheduler.get_last_lr()[0]:.2e} ETA={eta:.1f}min")
            
            # Checkpoint every CHECKPOINT_EVERY steps
            if global_step % CHECKPOINT_EVERY == 0:
                print(f"\n  ════ CHECKPOINT at step {global_step} ════")
                
                # Test generation
                test_prompt = "The meaning of life is"
                gen = test_generation(decoder, cse, field_mean, test_prompt)
                print(f"    Test: '{test_prompt}' → '{gen}'")
                
                # Save to GDrive
                metrics = {
                    'step': global_step,
                    'loss': accum_loss,
                    'avg_loss': avg_loss,
                    'samples_seen': idx + 1,
                }
                save_checkpoint_to_gdrive(decoder, optimizer, scheduler, global_step, metrics)
                
                if accum_loss < best_loss:
                    best_loss = accum_loss
                    print(f"    ★ New best loss: {best_loss:.4f}")
                
                print()
            
            accum_loss = 0.0
    
    except Exception as e:
        print(f"  ⚠ Error at sample {idx}: {e}")
        continue

# Training complete
print(f"\n  ════ TRAINING COMPLETE ════")
total_time = time.time() - start_time
final_avg_loss = total_loss / max(1, global_step)

print(f"    Total steps: {global_step}")
print(f"    Final avg loss: {final_avg_loss:.4f}")
print(f"    Best loss: {best_loss:.4f}")
print(f"    Time: {total_time/60:.1f} minutes")

log.metric("Total steps", global_step)
log.metric("Final loss", f"{final_avg_loss:.4f}")
log.cell_end("Cell 10 — Training Loop", "PASS")

## Cell 11: Final Upload to HuggingFace

In [ ]:
log.cell_start("Cell 11 — Final Upload")

print("\n" + "="*60)
print("  FINAL UPLOAD TO HUGGINGFACE")
print("="*60 + "\n")

final_metrics = {
    'total_steps': global_step,
    'final_loss': final_avg_loss,
    'best_loss': best_loss,
    'total_time_seconds': total_time,
    'samples_trained': len(all_texts),
    'datasets': dataset_sources,
}

save_final_to_apex_and_upload(decoder, apex, final_metrics, flx_path, hf_token)

log.cell_end("Cell 11 — Final Upload", "PASS")

## Cell 12: Comprehensive Generation Test Suite

In [ ]:
log.cell_start("Cell 12 — Generation Test")

print("\n" + "="*70)
print("  COMPREHENSIVE POST-TRAINING GENERATION TEST")
print("  Testing coherency across lengths and domains")
print("="*70 + "\n")

# ─────────────────────────────────────────────
# TEST SUITE: Diverse prompts with dynamic lengths
# ─────────────────────────────────────────────
# Format: (prompt, max_len, category, expected_style)

TEST_SUITE = [
    # ═══════════════════════════════════════════
    # SHORT RESPONSES (20-40 bytes) — Quick facts
    # ═══════════════════════════════════════════
    ("Question: What is 2 + 2?\nAnswer:", 25, "math", "single number"),
    ("Question: What color is the sky?\nAnswer:", 30, "qa", "one word"),
    ("Hello!", 20, "greeting", "short reply"),
    ("Yes or no:", 15, "binary", "yes/no"),
    
    # ═══════════════════════════════════════════
    # MEDIUM RESPONSES (50-100 bytes) — Sentences
    # ═══════════════════════════════════════════
    ("The meaning of life is", 60, "philosophy", "sentence"),
    ("Question: What is the capital of France?\nAnswer:", 50, "qa", "sentence"),
    ("Hello! How can I help you today?", 80, "assistant", "helpful reply"),
    ("The weather today is", 40, "completion", "description"),
    ("I think the best approach would be to", 70, "reasoning", "suggestion"),
    ("In my opinion,", 60, "opinion", "statement"),
    
    # ═══════════════════════════════════════════
    # LONGER RESPONSES (100-200 bytes) — Paragraphs
    # ═══════════════════════════════════════════
    ("Once upon a time in a magical forest,", 150, "story", "narrative"),
    ("To solve this problem, we need to first understand", 120, "reasoning", "explanation"),
    ("The history of artificial intelligence began", 130, "knowledge", "factual"),
    ("Here's how to make a great cup of coffee:", 140, "howto", "steps"),
    ("Dear friend, I wanted to share some thoughts with you about", 150, "letter", "personal"),
    
    # ═══════════════════════════════════════════
    # EXTENDED RESPONSES (200-400 bytes) — Testing limits
    # ═══════════════════════════════════════════
    ("Write a short story about a robot learning to feel emotions:", 250, "creative", "narrative"),
    ("Explain photosynthesis in simple terms:", 200, "education", "explanation"),
    ("Problem: A train leaves station A at 9am traveling 60mph. Another train leaves station B at 10am traveling 80mph. If the stations are 200 miles apart, when do they meet?\n\nSolution:", 300, "math", "step-by-step"),
    ("Human: I'm feeling sad today.\n\nAssistant:", 180, "emotional", "supportive"),
    
    # ═══════════════════════════════════════════
    # STRESS TEST (400+ bytes) — Coherency limits
    # ═══════════════════════════════════════════
    ("Chapter 1: The Beginning\n\nIt was a dark and stormy night when", 400, "novel", "extended narrative"),
    ("Let me explain the complete process of machine learning from start to finish:", 350, "technical", "long explanation"),
]

decoder.eval()
results = []

print("─" * 70)
for prompt, max_len, category, expected in TEST_SUITE:
    gen = test_generation(decoder, cse, field_mean, prompt, max_len=max_len)
    gen_len = len(gen.encode('utf-8'))
    
    # Simple coherency heuristics
    has_words = len(gen.split()) > 0
    has_repetition = any(word * 3 in gen.lower() for word in ['the', 'a', 'is', 'to', 'of', 'if', 'in'])
    word_count = len(gen.split())
    
    status = "✓" if has_words and not has_repetition else "⚠" if has_words else "✗"
    
    results.append({
        'category': category,
        'max_len': max_len,
        'actual_len': gen_len,
        'words': word_count,
        'status': status,
    })
    
    # Print with formatting
    prompt_display = prompt[:50] + "..." if len(prompt) > 50 else prompt
    prompt_display = prompt_display.replace('\n', '\\n')
    
    print(f"\n  [{category.upper()}] max={max_len} | expected: {expected}")
    print(f"  Prompt: '{prompt_display}'")
    print(f"  Output ({gen_len} bytes, {word_count} words): '{gen}'")
    print(f"  Status: {status}")
    print("─" * 70)

# ─────────────────────────────────────────────
# SUMMARY STATISTICS
# ─────────────────────────────────────────────
print("\n" + "="*70)
print("  GENERATION TEST SUMMARY")
print("="*70)

# Group by length category
short = [r for r in results if r['max_len'] <= 40]
medium = [r for r in results if 40 < r['max_len'] <= 100]
longer = [r for r in results if 100 < r['max_len'] <= 200]
extended = [r for r in results if 200 < r['max_len'] <= 400]
stress = [r for r in results if r['max_len'] > 400]

def calc_stats(group, name):
    if not group:
        return
    success = sum(1 for r in group if r['status'] == '✓')
    warning = sum(1 for r in group if r['status'] == '⚠')
    fail = sum(1 for r in group if r['status'] == '✗')
    avg_words = sum(r['words'] for r in group) / len(group)
    print(f"  {name}: {success}✓ {warning}⚠ {fail}✗ | avg {avg_words:.1f} words")

print("\n  By Length Category:")
calc_stats(short, "Short (≤40 bytes)   ")
calc_stats(medium, "Medium (41-100 bytes)")
calc_stats(longer, "Longer (101-200 bytes)")
calc_stats(extended, "Extended (201-400 bytes)")
calc_stats(stress, "Stress (400+ bytes) ")

# Overall
total_success = sum(1 for r in results if r['status'] == '✓')
total_tests = len(results)
print(f"\n  Overall: {total_success}/{total_tests} passed ({100*total_success/total_tests:.0f}%)")

# Coherency cliff detection
print("\n  Coherency Analysis:")
for i, r in enumerate(results):
    if r['status'] == '⚠' and i > 0 and results[i-1]['status'] == '✓':
        print(f"    ⚠ Coherency drops around {r['max_len']} bytes ({r['category']})")

log.metric("Tests passed", f"{total_success}/{total_tests}")
log.cell_end("Cell 12 — Generation Test", "PASS")

## Cell 13: Summary

In [ ]:
log.separator("Training Complete")

print("""
╔════════════════════════════════════════════════════════════════════╗
║              DECODER 50k TRAINING COMPLETE                         ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  DATASETS TRAINED:                                                 ║
║  ├── OpenWebText (10k) — General web text replay                  ║
║  ├── OpenAssistant (8k) — Helpful responses                       ║
║  ├── Anthropic HH-RLHF (8k) — Dialogues                          ║
║  ├── SQuAD (8k) — Question answering                             ║
║  ├── WritingPrompts (8k) — Storytelling                          ║
║  └── GSM8K (8k) — Reasoning                                       ║
║                                                                    ║
║  CHECKPOINTS:                                                      ║
║  ├── .pt files → Google Drive (every 5k steps)                    ║
║  └── .flx file → HuggingFace (final only)                         ║
║                                                                    ║
║  IMPROVEMENTS:                                                     ║
║  ✓ Word-boundary aware training                                   ║
║  ✓ Repetition penalty in generation                               ║
║  ✓ N-gram blocking                                                ║
║                                                                    ║
╚════════════════════════════════════════════════════════════════════╝
""")

print(f"\nFinal Statistics:")
print(f"  Samples trained: {len(all_texts)}")
print(f"  Total steps: {global_step}")
print(f"  Final avg loss: {final_avg_loss:.4f}")
print(f"  Checkpoints saved: {global_step // CHECKPOINT_EVERY}")
print(f"  Training time: {total_time/60:.1f} minutes")
print(f"\n  Checkpoint Storage: {CHECKPOINT_STORAGE}")
print(f"  HuggingFace: {HF_REPO_ID}")

log.success("Decoder 50k training complete!")